# 01 - Silver: ED Performance

**Pipeline:** Bronze -> Silver  
**Source:** AIHW MyHospitals API (measures MYH0005/0010/0011/0013)  
**Target:** `silver.fact_ed_performance` (Delta table)  

Run order: 01 -> 02 -> 03 -> 04

In [ ]:
WORKSPACE_ID = "e53f915a-de32-40a9-9b16-af4486796bbe"
LAKEHOUSE_ID = "6383e12e-91b9-4df2-99c5-06c9597bc27e"
ONELAKE_ROOT = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
SILVER_PATH  = f"{ONELAKE_ROOT}/Tables/silver/fact_ed_performance"
MEASURE_CODES = ["MYH0005", "MYH0010", "MYH0011", "MYH0013"]

from pyspark.sql.functions import col, lit, explode, size, to_date
from functools import reduce
from pyspark.sql import DataFrame

print(f"Root: {ONELAKE_ROOT}")

## Step 1 - Load Bronze Data

Reads raw JSON for each measure from the bronze layer (uploaded by `scripts/ingest_bronze.py`).  
Flattens the nested `reporting_unit_summary` struct.

In [ ]:
dfs = []
for code in MEASURE_CODES:
    path = f"{ONELAKE_ROOT}/Files/bronze/aihw/measures/{code}/raw.json"
    raw  = spark.read.option("multiline", "true").json(path)
    df   = raw.select(explode(col("result")).alias("r")).select(
        col("r.data_set_id"),
        col("r.value"),
        col("r.lower_value"),
        col("r.upper_value"),
        col("r.suppressions"),
        col("r.reporting_unit_summary.reporting_unit_code").alias("hospital_code"),
        col("r.reporting_unit_summary.reporting_unit_name").alias("hospital_name"),
        col("r.reporting_unit_summary.reporting_unit_type.reporting_unit_type_code").alias("unit_type")
    ).withColumn("measure_code", lit(code))
    dfs.append(df)
    print(f"{code}: {df.count():,} rows")

combined = reduce(DataFrame.unionByName, dfs)
print(f"Total: {combined.count():,} rows")

## Step 2 - Profile Schema

Print schema and sample rows to confirm data landed correctly.

In [ ]:
combined.printSchema()
combined.show(5, truncate=False)

## Step 3 - Filter to WA + Join Time Periods

Two joins to add context not returned by the data-items endpoint:
1. Join with `wa_hospitals.json` to keep only WA hospitals
2. Join with `datasets.json` on `data_set_id` to get `time_period_start` / `time_period_end`

Suppressed records (non-empty `suppressions` array) are excluded.

In [ ]:
wa_raw = spark.read.option("multiline", "true").json(
    f"{ONELAKE_ROOT}/Files/bronze/aihw/reporting_units/wa_hospitals.json"
)
wa_codes = (
    wa_raw.select(explode(col("result")).alias("h"))
    .select(col("h.reporting_unit_code").alias("hospital_code"))
)
print(f"WA hospital codes: {wa_codes.count()}")

ds_raw = spark.read.option("multiline", "true").json(
    f"{ONELAKE_ROOT}/Files/bronze/aihw/datasets/datasets.json"
)
datasets = (
    ds_raw.select(explode(col("result")).alias("d"))
    .select(
        col("d.data_set_id"),
        to_date(col("d.reporting_start_date")).alias("time_period_start"),
        to_date(col("d.reporting_end_date")).alias("time_period_end")
    )
)
print(f"Dataset records: {datasets.count()}")

silver_df = (
    combined
    .filter(col("unit_type") == "H")
    .join(wa_codes, "hospital_code", "inner")
    .join(datasets, "data_set_id", "left")
    .select(
        col("hospital_code"),
        col("hospital_name"),
        col("measure_code"),
        col("time_period_start"),
        col("time_period_end"),
        col("value").cast("double"),
        col("lower_value").cast("double"),
        col("upper_value").cast("double"),
        (size(col("suppressions")) > 0).alias("suppressed")
    )
    .filter(col("value").isNotNull())
    .filter(col("time_period_start").isNotNull())
    .filter(col("suppressed") == False)
)

print(f"Final WA silver rows: {silver_df.count():,}")
silver_df.show(10)

## Step 4 - Data Quality Checks

Validate MYH0005 values are in `[0, 100]`, hospital and measure counts are non-zero, and date range looks correct.

In [ ]:
invalid = (
    silver_df.filter(col("measure_code") == "MYH0005")
             .filter((col("value") < 0) | (col("value") > 100))
             .count()
)
print(f"MYH0005 values outside [0,100]: {invalid}")
print(f"Distinct hospitals: {silver_df.select('hospital_code').distinct().count()}")
print(f"Distinct measures:  {silver_df.select('measure_code').distinct().count()}")

min_date = silver_df.agg({"time_period_start": "min"}).collect()[0][0]
max_date = silver_df.agg({"time_period_start": "max"}).collect()[0][0]
print(f"Date range: {min_date} to {max_date}")

## Step 5 - Write Silver Delta Table

Writes to OneLake and registers as `silver.fact_ed_performance`.  
Mode: `overwrite` - safe to re-run.  
Next: run `02_silver_dim_hospitals`, then `03_gold_ed_trends`.

In [ ]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_PATH)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS silver.fact_ed_performance
    USING DELTA
    LOCATION '{SILVER_PATH}'
""")

print("silver.fact_ed_performance written successfully")
print(f"Location: {SILVER_PATH}")
print(f"Row count: {spark.table('silver.fact_ed_performance').count():,}")